# 📓 Llama-Index Practice

In this section, we will create a simple Llama Index app and learn how to log and get feedback on an LLM response. Additionally, we will explore how the query engine can be modified based on its intended purpose.​

Through this process, we hope to gain a clear understanding of the role and functionalities of the Llama Index, as well as the specific inputs and outputs of each method.  

<img src="https://miro.medium.com/v2/resize:fit:1400/0*CrywD0tloiK9dgy_.png" width="600">

This section is divided into three main stages:

###I. Build a Query Engine  
###II. Use Database Management Method  
###III. Make a Custom Query Engine

Let's first look at what is needed to build a Query Engine.

## I. Buliding a Query Engine

The task of this section is to create a Query Engine that takes a user's query, retrieves for related content, and returns a final summary.  

Ignore the `Vector DB` in the picture. We don't use external vector database in the pracetice. Instead, we will use local memory as simple vector store.


<img src="https://i.imgur.com/bR4xaBd.png">

To accomplish this task, the following steps will be taken.
<br/>

#### 1. Install and Import Libraries
#### 2. DownLoad Data
#### 3. Make Query Engine from Index Directly
#### 4. Conduct Activity


### 1. Install and Import Libraries

In [1]:
### YOUR CODE HERE ###
import os
os.environ.pop("PYTHONPATH", None)

! pip install llama_index==0.12.2 openai==1.55.3 packaging>=24.0 streamlit==1.35.0 --quiet

In [2]:
### YOUR CODE HERE ###

import os
os.environ["OPENAI_API_KEY"] = "" #Insert your openai api key

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, PromptTemplate
from openai import OpenAI

from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

import textwrap
import openai

C:\Users\swsuser-j06\.conda\envs\rag\lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


### 2. DownLoad Data

Previously, we learned that the Query Engine receives the following elements as input and output.
<br/>
#### Input: **User Query**, **Vector Database**
#### Output: **Summary of Retrieved Passages**
<br/>

Here, the User Query is an element that we can decide. However, since the Vector Database must contain information related to the Query, we need to consider how to configure the Vector Database before proceeding with the practice.

The Vector Database can be divided into two categories: a storage that allows the query engine to retrieve related passages, and the data containing the related passages. In this practice, we will use local memory as a data storage instead of an external Vector Database, so the Vector Database will be referred to as a Vector Store from now on. For the data, we will use a simple txt file provided by Llama Index.

Before we begin the practical exercises, let's first take a look at the txt file we used. The code below can be used to download the txt file we used with the `wget` command.


In [3]:
### YOUR CODE HERE ###

import os, urllib.request

os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt"
urllib.request.urlretrieve(url, "data/paul_graham_essay.txt")

('data/paul_graham_essay.txt', <http.client.HTTPMessage at 0x20bfa8af850>)

If there are no issues with the download, you should be able to see that the `paul_graham_essay.txt` file has been downloaded.

This example uses the text of Paul Graham’s essay, [“What I Worked On”](https://paulgraham.com/worked.html), and is the canonical llama-index example.


Let’s print the first fifteen lines of the txt file. This will allow us to see that the text file deals with the experiences and reflections of one person's life.


In [4]:
import os
print(os.getcwd())

C:\workspace\05_RAG\1일차\실습 자료\Code


In [7]:
### YOUR CODE HERE ###

path_to_txt = r'c:\workspace\05_RAG\1일차\실습 자료\Code\data\paul_graham_essay.txt' #change this path

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for i in range(15):
        line = file.readline()
        if not line:
            break
        print(line.strip())
print("\n...")



What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t

### 3. Make Query Engine from Index Directly

Now let's create a query engine unsing the above information.

First, since Llama Index cannot directly handle text files in their raw form, we must convert them into Documents and Nodes. This can be accomplished with the following code.

```Python
documents = SimpleDirectoryReader("data").load_data()
```

SimpleDirectoryReader takes the directory path containing data that will go into the Vector Store as input and converts all the files within that directory into Document objects. The data types that can be converted into Documents are as follows.


*   .txt - text file  
*   .csv - comma-separated values  
*   .docx - Microsoft Word  
*   .epub - EPUB ebook format  
*   .hwp - Hangul Word Processor  
*   .ipynb - Jupyter Notebook  
*   .jpeg, .jpg - JPEG image  
*   .mbox - MBOX email archive  
*   .md - Markdown  
*   .mp3, .mp4 - audio and video  
*   .pdf - Portable Document Format  
*   .png - Portable Network Graphics  
*   .ppt, .pptm, .pptx - Microsoft PowerPoint  

You can view the text contained in the `documents` using the following command:

```Python
for i in range(len(documents)):
  print(documents[i].text)
```

In [10]:
### YOUR CODE HERE ###

# 지정된 dir에서 모든 파일 읽음
documents = ???????

In [11]:
### YOUR CODE HERE ###

for i in range(len(documents)):
  print(documents[i].text)



What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t

Once the Document is complete, we need to use this data to create an index.

The index is created using the VectorStoreIndex.from_documents() method. This method takes a list of Node objects or a Document as input.


```Python
index = VectorStoreIndex.from_documents(documents)
```

If connected to an external vector database, Llama Index follows the index algorithm of that vector database. If there is no connection to an external vector database, it stores the data in the local memory in the form of a dictionary.

In [12]:
### YOUR CODE HERE ###

# Documents 를 node 단위로 잘라서 embedding 함
index = ???????

Now, let’s look at how the index is structured.  

Our index is built based on documents. However, as mentioned in the previous presentation, when Llamaindex stores data in the database, it breaks down the document into nodes. Therefore, let’s first check if the document has been divided into nodes.

In [13]:
### YOUR CODE HERE ###

node_id = ???????

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

d9f4ee0d-3ed1-4e76-9817-e565f37b1d45 d9f4ee0d-3ed1-4e76-9817-e565f37b1d45
The number of nodes:  22
dict_values(['d9f4ee0d-3ed1-4e76-9817-e565f37b1d45', 'bde4c15e-1988-4b39-ad78-12868dce4d33', '76a7b344-637d-4483-829e-c627f8598bf6', 'dc9cbc81-c9c1-43a1-83bd-b0c9bc701ade', '003955e6-850c-404e-9f1a-5e64101f7a19', '5e21cf40-c649-457b-b942-bcce48448fe5', '49f5576b-d899-44e5-bdc1-deee07f211a0', '846b82f6-dd80-440c-9d47-20afb94cfe87', '791021d7-01a4-44b5-9297-d498afeb51bd', 'fe3cb8dd-5322-4807-8523-43161b4c5c02', '3dae4b23-6584-4500-87bb-52488a1d8c9a', '8eff6c1c-ea09-4678-872c-57e1652a7793', 'd032de9b-716b-417f-9572-717114ef96d9', '5a47a916-c511-451d-aa8e-f97c1fa8af7f', 'abcaf837-b013-4dd2-afd0-f2319769134f', 'ca912f07-6c0a-4cf7-b93c-8c5cc84441a6', 'acd3fd4b-9538-4938-a37d-00a0b6a54114', 'e63bbb93-765b-466e-84d5-0dc15c6acf50', 'ffd93303-0620-469d-98e6-824375829f1b', '1f3c2384-3c87-4352-b82c-5642b7a5c117', '67f570cc-e71e-4c8e-bfdd-76bd61a5f789', '813050bf-2b83-4f0d-aff2-159b05612778'])


By looking at this, we can check the id of the first node and the number of nodes the document has been divided into.

Next, let’s see how each node has been transformed into embeddings.


In [16]:
### YOUR CODE HERE ###

index.vector_store.data.embedding_dict[node_example_id]

# 22개 node가 1536차원data임
print(len(index.vector_store.data.embedding_dict[node_example_id]))

1536


As you can see here, each node is automatically converted into embeddings, becoming vectors when the index is created.

Next, let’s verify why the total number of nodes is as shown. When splitting a document into nodes, Llamaindex uses a class called SentenceSplitter. This class allows us to divide the text data into predefined lengths.

In [17]:
### YOUR CODE HERE ###

from llama_index.core.node_parser import SentenceSplitter

parser = ???????

nodes = ???????

print(len(nodes))

22


Each node has an attribute called `text`, allowing us to see what text data each node contains. Additionally, you can see that the texts overlap by the value of the `chunk_overlap`.

When checking the length of each text data, we can see that they are composed of different numbers of characters. This is because the `SentenceSplitter` class divides `documents` not by the number of characters but by tokens, which are a type of sub-word.

In [21]:
### YOUR CODE HERE ###

# 각 node들의 text에 접근
print(nodes[0].text)
print("\n----------------------\n")
print(nodes[1].text)
print("\n----------------------\n")
# character 단위로 갯수 샘. Chunk 단위로 하면 1024가 됨 # chunk: 긴 문서를 모델이 읽기 좋은 크기로 임의로 자른 덩어리
print(len(nodes[0].text), len(nodes[1].text))

What I Worked On

February 2021

Before college the two main things I worked on, outside of school, were writing and programming. I didn't write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.

The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district's 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in the

We can use the default tokenizer used by Llamaindex’s `SentenceSplitter` to directly check how many tokens each text consists of.

By running the code below, we can bring in the tokenizer of `gpt-3.5-turbo`, which is used by default in `SentenceSplitter`, and perform tokenization to see the results and check how many tokens are generated.

When we look at the tokenization results, we will see that they differ from the `chunk_size` we set. This is because, when creating nodes, the amount of text data to be tokenized is determined by considering the metadata as well, in order to compose nodes of consistent size including the amount of metadata.

In [24]:
### YOUR CODE HERE ###

import tiktoken
tokenizer = ???????("gpt-3.5-turbo")

encoded_text = ???????(nodes[0].text)
print(f"Encoded text: {encoded_text}")
print(f"Encoded text length: {len(encoded_text)}")

decoded_text = ???????(???????)
print(f"Decoded text: \n\n{decoded_text}")

Encoded text: [3923, 358, 5664, 291, 1952, 271, 33877, 220, 2366, 16, 271, 10438, 7926, 279, 1403, 1925, 2574, 358, 6575, 389, 11, 4994, 315, 2978, 11, 1051, 4477, 323, 15840, 13, 358, 3287, 956, 3350, 23691, 13, 358, 6267, 1148, 7314, 16483, 1051, 10171, 311, 3350, 1243, 11, 323, 4762, 2103, 527, 25, 2875, 7493, 13, 3092, 7493, 1051, 25629, 13, 2435, 1047, 20781, 904, 7234, 11, 1120, 5885, 449, 3831, 16024, 11, 902, 358, 35706, 1903, 1124, 5655, 382, 791, 1176, 7620, 358, 6818, 4477, 1051, 389, 279, 29022, 220, 6860, 16, 430, 1057, 2978, 9474, 1511, 369, 1148, 574, 1243, 2663, 330, 695, 8863, 1210, 1115, 574, 304, 220, 24, 339, 12239, 11, 779, 358, 574, 220, 1032, 477, 220, 975, 13, 578, 2978, 9474, 596, 220, 6860, 16, 7077, 311, 387, 304, 279, 31741, 315, 1057, 27144, 1579, 2978, 11, 323, 856, 4333, 8269, 2999, 4798, 323, 358, 2751, 8041, 311, 1005, 433, 13, 1102, 574, 1093, 264, 13726, 24537, 40148, 596, 1208, 404, 1523, 1070, 11, 449, 682, 1521, 20167, 31348, 12933, 2001, 14266, 11

You can see that the result here matches the number we saw earlier.  

If you want to change the default `chunk_size` or `chunk_overlap` when you make `index`, you can use `transformations` argument in `VectorStoreIndex.from_documents()`.


In [25]:
### YOUR CODE HERE ###

# chunk 사이즈를 200으로 지정
text_splitter = ???????

index = ???????

In [26]:
### YOUR CODE HERE ###

node_id = index.index_struct.nodes_dict

for key, value in node_id.items():
    print(key, value)
    node_example_id = value
    break

print("The number of nodes: ", len(node_id.values()))

print(node_id.values())

# chunk사이즈를 200으로 할 경우, 147개 node로 잘라짐

edb04ed9-9bf5-438b-b62e-878781405eb1 edb04ed9-9bf5-438b-b62e-878781405eb1
The number of nodes:  147
dict_values(['edb04ed9-9bf5-438b-b62e-878781405eb1', '66e246fd-150a-4875-84e4-a7b5ca41dec0', '50f62005-6f35-49ec-b766-de3d039dcf22', 'a733a097-937c-4d41-8283-322e9b64512d', '20cf07ac-a9c4-4702-bb1b-8d2774895872', 'aaff6127-7577-46bb-99aa-712c9f6fc3ec', '2bbd9627-c974-4798-8280-09b9d6d505bd', '24e31fa1-dc40-4558-a3ed-38dd5b8fad42', '9619d597-8e64-440a-846a-599a8561ab0c', 'a8cdd293-01a2-4d66-b3bf-df9923271d38', 'e45d28d3-3363-4704-8162-22c032045887', 'cad7b095-b695-486a-b23c-41a532481c01', 'c02b71ba-e015-471e-9900-8e32398eadc6', '8ad6d5fe-5008-404b-bf60-45340000a314', 'abeef54b-6aed-485b-bd5c-1446cada6d0f', '4a91c918-f40f-479c-8715-8e1200ba12d4', 'b7e7eb24-1158-4bee-90a5-188ebbb3637f', '435ddebe-aa25-4ef8-b652-bb4f73f96061', '7f6b7e38-8a50-4be6-8f98-89a62ff839fb', '57fb34be-a594-493d-98ab-0d0c74399d24', 'a4c4554a-6b2c-42b1-b8a8-0986d4742606', 'dbc0593c-6af6-4ac6-bca2-f2c0d63d5573', '2304e6

But, in this practice, we will use default `SentenceSplitter`.

In [27]:
### YOUR CODE HERE ###

###  important
index = ???????

Once the index is created, Llama Index provides a way to immediately create a query engine using the index. With the following code, we can create a query engine using the index.

In [29]:
### YOUR CODE HERE ###

### important
query_engine = ???????

### 4. Conduct Activity

Now let's actually use the query engine. We will check the results of query engine using the following questions.

#### 4-1. Answerable Questions from Data

In this activity, we will check how query engine will answer **a question related to the given data, and the information needed for the correct answer is directly provided in the data**.

#### 4-2. Unrelated Question to the Data

In this activity, we will check how query engine will answer **a question that has absolutely nothing to do with the given data**.

#### 4-3. Question Requiring Complex Reasoning

In this activity, we will check how query engine will answer **a question that is hard to answer, which means that question is related to the given data, but the information needed for the correct answer is not directly given in the data**.

### 4-1. Answerable Questions from Data
First, let's ask the query engine some questions that it can answer and some that it cannot.

For example, we could ask what program the author of this data has written about. The answer to this question is in the second sentence of the txt file.

#### Question: **What is the first programs the author tried writing?**

#### In paul_graham_essay: **The first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing."**

Intuitively, it is expected that the query engine will generate the correct answer, which is IBM 1401. Let's verify that.

In [30]:
### YOUR CODE HERE ###

text_chunk = "The first programs I tried writing were on the IBM 1401 that our school district used for what was then called \"data processing.\""

if text_chunk in documents[0].text:
    print("True")
else:
    print("False")

True


In [31]:
### YOUR CODE HERE ###

response = ???????("What is the first programs the author tried writing?")

print(response)

The first programs the author tried writing were on the IBM 1401 using an early version of Fortran.


<img src="https://xe.obg.co.kr/files/attach/images/4199/352/004/7e2189cf45a27f9b9cda5fef28c1dd5f.gif">

As expected, the results are accurate.  


### 4-2. Unrelated Question to the Data

This time, We will check whether the query engine performs its intended function well.  
A query engine is **a generic interface that allows users to ask questions about external data**.  
Therefore, **it should not be able to answer questions that cannot be answered through external data**.

The reason is that if the query engine can answer questions that are completely unrelated to the external data, it implies that the query engine can freely use parameterized knowledge. This, in turn, means that the answers from the query engine may contain information that is incorrect or not up-to-date.

<img src="https://media0.giphy.com/media/ANbD1CCdA3iI8/200w.gif?cid=6c09b952ooe7fzryithl047ty5npdt1xd50tlu9gcpqnzh87&ep=v1_gifs_search&rid=200w.gif&ct=g" width="200" height="200"><img src="https://st2.depositphotos.com/4421345/11492/v/950/depositphotos_114921728-stock-illustration-public-speaking-robot.jpg" width="200">

So, let’s pass on a question that has nothing to do with the original data but that the query engine’s LLM can answer.  
For example, we could use a question like this.

#### Question: **How many countries participated in the production of the space station?**
#### Answer: **15**

In [34]:
### YOUR CODE HERE ###

response = ???????("How many countries participated in the production of the space station?")

print(response)

There is no information provided in the context about the number of countries that participated in the production of the space station.


<img src = "https://img.danawa.com/images/descFiles/6/164/5163558_1_16652362092043267.gif
" width = 200>

It doesn't produce the correct answer as expected. However, we can find what kinds of LLM used in query engine through the official LlamaIndex documentation.

Actually, Llamaindex uses `gpt-3.5-turbo` as LLM of response synthesizer. So, we can ask `gpt-3.5-turbo` to answer the question.

Let's check this out with a simple code. We can use the following code to separately utilize the OpenAI LLM.  



In [35]:
### YOUR CODE HERE ###

def generate_answer(question):
    messages = ???????
    response = ???????

    return response.???????

In [36]:
### YOUR CODE HERE ###

question = "How many countries participated in the production of the space station?"

print(generate_answer(question))

The International Space Station (ISS) is a collaborative project involving five space agencies from 15 countries. These countries are the United States, Russia, Japan, Canada, and 11 European countries represented by the European Space Agency.


In fact, as in the example above, OpenAI LLM already knows the answer to this question.  
However, since the relevant information cannot be found in the txt file that query engine can refer to here, it does not generate a correct answer for a question that is already known.  

<img src="https://i.imgur.com/sxSRQks.png">

From this, we can understand that LLM in the response synthesizer of the query engine uses the information from the retrieved passages produced by the retriever, excluding its own parametrized knowledge.  

### 4-3. Question Requiring Complex Reasoning

Now, let’s explore more deeply. There is a question related to the data that is hard to answer. We will see how the query engine responds to such questions.

The question we will ask in this section is who is the author of the `paul_graham_essay.txt`. Becuase the original text file does not mention the author's name directly, this is a complex question.

Question: **Who is the author?**

Let's see whether the writer's name is actually not in the original text file or not.

In [37]:
### YOUR CODE HERE ###

wrapper = textwrap.TextWrapper(width=80)

with open(path_to_txt, 'r', encoding='utf-8') as file:
    for line in file:
        if 'paul graham' in line.lower():
            formatted_text = wrapper.fill(line.strip())
            print(formatted_text.split('.')[-1])


Occasionally after wrestling for hours with some gruesome bug I'd check Twitter
or HN and see someone asking "Does Paul Graham still code?"


This is the only sentence in the original txt file that mentions 'Paul Graham'.  
Now, let's read it ourselves and consider whether it actually specifies the author's name.

<br/>

Someone might infer that the author's name is Paul Graham from this sentence, but I think it's not certain

In my opinion, this sentence does not explicitly state the author's name, so it does not provide enough useful information

<br/>

However, opinions on this may vary, so let's verify it ourselves.


In [38]:
### YOUR CODE HERE ###

response_complex = ???????("Who is the author?")

print(response_complex)

The author is Paul Graham.


<img src="https://i3.ruliweb.net/ori/21/07/30/17af799f631524cc2.gif">

Hmm... The query engine answers better than we think.  
Then, let's see what data were referred to by the query engine.
<br/>

<img src="https://i.imgur.com/sxSRQks.png">

The query engine made using `as_query_engine` uses a retriever generated directly from the index by using `as_retriever` method.   
Therefore, if we look at the results of this retriever, we can see which retrieved passages the query engine has seen.


In [39]:
### YOUR CODE HERE ###

retriever = ???????()

ret_passages = ???????("Who is the author?")

for i in range(len(ret_passages)):
  print("###Retrieved Passage\n", ret_passages[i].text)
  print("\n\n\n")

###Retrieved Passage
 All that seemed left for philosophy were edge cases that people in other fields felt could safely be ignored.

I couldn't have put this into words when I was 18. All I knew at the time was that I kept taking philosophy courses and they kept being boring. So I decided to switch to AI.

AI was in the air in the mid 1980s, but there were two things especially that made me want to work on it: a novel by Heinlein called The Moon is a Harsh Mistress, which featured an intelligent computer called Mike, and a PBS documentary that showed Terry Winograd using SHRDLU. I haven't tried rereading The Moon is a Harsh Mistress, so I don't know how well it has aged, but when I read it I was drawn entirely into its world. It seemed only a matter of time before we'd have Mike, and when I saw Winograd using SHRDLU, it seemed like that time would be a few years at most. All you had to do was teach SHRDLU more words.

There weren't any classes in AI at Cornell then, not even graduate c

This is quiet lone passages. So, it is hard to check whether there is any evidence of query engine's response or not.  
Can you find any supporting evidence of the question?  

Let's see if 'Paul Graham' is in there.



In [40]:
### YOUR CODE HERE ###

passages = ''

for i in range(len(ret_passages)):
  passages += ???????

if 'paul graham' in passages.lower():
    print("The word 'Paul Graham' is found in the passages.")
else:
    print("The word 'Paul Graham' is not found in the passages.")

The word 'Paul Graham' is not found in the passages.


<img src="https://i2.ruliweb.net/ori/21/07/24/17ad8e756ba54811a.gif" width="400">

???

Retriever couldn't find the only sentence with the word Paul Graham on it. Then how did query engine know that the answer was Paul Graham?

To verify this, let’s using the query engine’s LLM as we did in the previous section and ask the same question. We will use a prompt as similar as possible to the one used in LlamaIndex response synthesizer, but with an additional instruction to provide the reason for the LLM’s answer.  

The prompt used by the original Response Synthesizer is as follows.


<img src = "https://i.imgur.com/ZXUDAJi.jpeg" width="450">  


<br/>

Therefore, we will write the code as shown below to check the results of the LLM.



In [41]:
### YOUR CODE HERE ###

ret_context = ""
for ret_result in ret_passages:
  ret_context += ret_result.text

question = ???????

print(generate_answer(question))

The author of the text is Paul Graham. This can be inferred from the personal experiences and insights shared throughout the text, such as working on AI, writing a book about Lisp hacking, starting a company to put art galleries online, and founding Viaweb.


Well... it seems like query engine can **capture useful information not only in direct evidence but also in indirect content by using its prior knowledge, i.e., parametrized knowledge**.  

Therefore, we can consider that the query engine has the ability to provide correct answers even when the given query is not explicitly answered in the data.  

However, the problem is that **without such an explanation, the reasoning process cannot be trusted** because it is impossible to know whether the LLM used reliable information in its reasoning.

<br/>

Here’s what we can infer from this:

the query engine has the capability to answer not only questions directly addressed in the data **but also questions that can be indirectly inferred from the data.**  
However, in such cases, **the reliability of the answers is compromised because we cannot verify the information used and the reasoning steps.**   

To address this issue, we can use one of the following two methods.  


#### **1.** Force the response synthesizer to utilize only the information in a given retrieved passages.
#### **2.** When the response synthesizer generates a summary, make sure that the reason is also generated.

## II. Use Database Management Method

Now, let’s consider a scenario. If we have new data and want to create a query engine using it, we could recreate the `document`, `index`, and `query engine` as we did above.

However, this is quite cumbersome and time-consuming. In addition, if you want to insert new data so that query engine uses that data, the above method is very expansive because a new index must be generated for every inserts operation.

So, Llama Index is providing the following functions to manage data smoothly.

### 1. Insert

We can insert new data to the existing query engine.

To check this, we will ask a question which can't get useful information from the data before inserting new data, and see what happens after inserting data.

In this section, the following data and question will be used.

Inserted Data: **"Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."**  
Question: **What are the temperature and air pressure conditions under which natural diamonds are produced?**


In [42]:
### YOUR CODE HERE ###

data_text = "Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa."
question = "What are the temperature and air pressure conditions under which natural diamonds are produced?"

res = query_engine.query(question)

print(res)

Natural diamonds are typically formed under high temperature and pressure conditions deep within the Earth's mantle.


<img src = "https://i.pinimg.com/originals/15/8b/ed/158bed9819e4fccf7e18a5eeeaf79c6b.png" width = 200>

Hmm… the query engine answers quite well. It seems the response synthesizer is using its parameterized knowledge. However, there are no specific details like 900–1,400°C and pressures of 5–6 GPa.

Anyway, it's time to insert the data.



In [43]:
### YOUR CODE HERE ###

docu = ???????

???????

new_query_engine = ???????

In [44]:
### YOUR CODE HERE ###

new_res = ???????

print(new_res)

Natural diamonds are formed in metallic melts at temperatures of 900–1,400 °C and at pressures of 5–6 GPa.


It's as expected. Query engine is generating an appropriate answer with detail using information in new `Document` object.

Then, what happens if we change the information in that data?

### 2. Update

If a Document is already present within an index, you can "update" a `Document` with the same doc `id_`. The `update_ref_doc` method receives a single `Document` object and updates the value of data that has the same `id`.

Or you can "refresh" all document at once. The `refresh_ref_doc` method finds the `id` of the `Document` object that came into the input, updates the data with the same `id`, or inserts the data if there is no other data with the same `id`

**`update_ref_doc`**  
**Input**: single `Document` object  
**Output**: None

**`refresh_ref_doc`**  
**Inpu**t: list of `Document` object  
**Output**: a boolean list, indicating which documents in the input have been refreshed in the index.

You will see `False` in that boolean list if text of document doesn't change.  

In both methods, we can set `delete_from_docstore` to `True` or `False`. A detailed description of this will be given in Delete.


<br/>

Let's check by changing the base question a little bit.  

I changed the information about temperature and air pressure in the question, so check it out for yourself.  



In [45]:
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 2,000–6,000 °C and at pressures of 8–9 GPa.")

output = ???????

print(output)

query_engine_update = ???????

res_update = ???????

print(res_update)

None
Natural diamonds are formed in metallic melts at temperatures ranging from 2,000 to 6,000 degrees Celsius and at pressures between 8 to 9 gigapascals.


In [46]:
### YOUR CODE HERE ###

docu.set_content(value="Natural diamonds were (and are) formed (thousands of million years ago) in the upper mantle of Earth in metallic melts at temperatures of 6,000–8,000 °C and at pressures of 12–15 GPa.")

output = ???????

print(output)

query_engine_refresh = ???????

res_refresh = ???????

print(res_refresh)

[True]
Natural diamonds are formed in metallic melts at temperatures ranging from 6,000 to 8,000 degrees Celsius and at pressures between 12 to 15 gigapascals.


You can check list of doc_id in the index. So, if you want more experience with other data, you can check whether your data is inserted properly by looking at that list

In [47]:
### YOUR CODE HERE ###

print(index.???????)

dict_keys(['d857f2b6-c67c-4c17-b7c9-52ee455e14c0', 'new_doc_id'])


### 3. Delete

We wrapped the new information above with a `Document` object and inserted it into the index.

At this time, we were able to specify the `Document` object using `doc_id`. What this doc_id is can be confirmed as follows.

In [48]:
### YOUR CODE HERE ###

id = ???????

print(id)

new_doc_id


Using this, we can change the contents of data with the `doc_id` using the `delete` method of the index.

Let's delete the data related to a diamond, and see how query engine answers for the same question.

Through the value of `delete_from_docstroe`, it is possible to determine whether the data with that id actually disappears on the database, or disappears only on the index and remains in the database. Even if it disappears only on the index, the information of the data cannot be used by the query engine.



In [49]:
### YOUR CODE HERE ###

index.???????

query_engine_delete = ???????

res_delete = ???????

print(res_delete)

Natural diamonds are typically formed under high temperature and pressure conditions deep within the Earth's mantle.


If the answer still contains very details of condition of natural diamonds, check the index whether that data was deleted properly.

The result of `index.ref_doc_info.keys()` has to contain only one doc_id.

In [50]:
### YOUR CODE HERE ###

print(index.???????)

dict_keys(['d857f2b6-c67c-4c17-b7c9-52ee455e14c0'])


## III. Make a Custom Query Engine

In this section, we will define the existing query engine in more detail and modify its internal components to create the query engine we desire.
<br/>

The query engine can vary in its usage depending on the task requested after passing the retrieved documents to the LLM in response synthesizer. For instance, it can be tasked with summarizing the documents or directly answering the query.

Since the query engine can read through extremely long documents that exceed the LLM’s context limit, it can have a significant impact on RAG, depending on how it is used. Therefore, choosing which query engine to use is an important design choice when designing RAG.

This section is composed of the following four stages:
 <br/>
#### 1. Defining the query engine in detail
#### 2. Creating a custom query engine


### 1. Defining the query engine in detail

In this section, we will define the retriever and response synthesizer, which are internal components of the query engine, and use them to define a class that performs the role of the query engine.

In [64]:
### YOUR CODE HERE ###

class StandardQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        response_obj = self.response_synthesizer.synthesize(query_str, nodes)
        return response_obj

Initialize the retriever and response synthesizer, and then verify that the class functions correctly.


In [65]:
### YOUR CODE HERE ###

retriever = index.???????
synthesizer = ???????
query_engine = ???????

response = query_engine.custom_query("What did the author do growing up?")
print(str(response))

The author worked on writing and programming before college.


### 2. Creating a custom query engine

Now, let’s modify the query engine so that we can pass the desired task to the response synthesizer through a prompt.  

While there are many aspects of the retriever that can also be adjusted, in this exercise, we will focus on changing the content passed to the response synthesizer to experience more noticeable changes

from llama_index.core import PromptTemplate


In [66]:
### YOUR CODE HERE ###

simple_qa_prompt = ???????
short_sum_prompt = ???????

In [67]:
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class OurCustomQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OpenAI
    qa_prompt: PromptTemplate = simple_qa_prompt

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

Now, let’s check if it truly works as we intended.


In [71]:
### YOUR CODE HERE ###

llm = OpenAI(model="gpt-3.5-turbo")

query_engine = OurCustomQueryEngine(
    ???????,
    ???????,
    ???????,
    ???????,
)

response = query_engine.custom_query("What did the author do growing up?")
print(str(response))

The author worked on writing short stories and programming, starting with the IBM 1401 in 9th grade and later moving on to microcomputers like the TRS-80.


In [72]:
### YOUR CODE HERE ###

llm = OpenAI(model="gpt-3.5-turbo")

query_engine = OurCustomQueryEngine(
    ???????,
    ???????,
    ???????,
    ???????,
)

response = query_engine.custom_query("What did the author do growing up?")
print(str(response))

The author worked on writing short stories and programming, starting with an IBM 1401 in 9th grade.
